In this notebook the model will be chosen and after defyining the threshold i will create full pipeline that works with the raw data

In [60]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (accuracy_score, roc_auc_score, average_precision_score,
    precision_recall_curve, confusion_matrix,
    precision_score, recall_score, f1_score, classification_report)

from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.base import clone



from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    cross_val_predict
)

import pandas as pd
import numpy as np  
from pathlib import Path
import os
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
import joblib
import json

In [81]:
BASE_PATH= Path(os.getcwd()).parent

DATASET_PATH= BASE_PATH / 'dataset'
TRAIN_PATH= DATASET_PATH / 'split' /  'train_set.csv'
TEST_PATH= DATASET_PATH / 'split' /  'test_set.csv'

ARTIFACTS_PATH= BASE_PATH / 'artifacts'
FEATURES_PATH= ARTIFACTS_PATH / 'final_features.json'
MODEL_DATA_PATH=ARTIFACTS_PATH / 'model_data'
THRESHOLD_PATH=MODEL_DATA_PATH/'threshold_recall.json'


In [8]:
train_df=pd.read_csv(TRAIN_PATH)
test_df=pd.read_csv(TEST_PATH)

In [26]:
with open(FEATURES_PATH, "r", encoding='utf-8') as f:
    meta=json.load(f)

In [28]:
target=meta['target']
features=meta['final_features']
random_state=meta['random_state']
PAY_N=meta['PAY_N']
PAY_AMT=meta['PAY_AMT']
BILL_AMT=meta['BILL_AMT']

In [49]:
def make_preprocessor(X: pd.DataFrame) -> ColumnTransformer:
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    ord_cols=PAY_N
    cat_cols = [c for c in features if c not in num_cols and c not in ord_cols]
    ordinal_categories = [[-2, -1, 0, 1, 2, 3]] * len(ord_cols)
    num_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler(with_mean=True))
    ])
    ord_pipe= Pipeline(steps=[
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("ordenc", OrdinalEncoder(
                categories=ordinal_categories,
                handle_unknown="use_encoded_value",
                unknown_value=-1
            ))
        ])
    cat_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", num_pipe, num_cols),
            ("ord", ord_pipe, ord_cols),
            ("cat", cat_pipe, cat_cols)
        ],
        remainder="drop"
    )
    return preprocessor


def make_pipeline(model, X: pd.DataFrame) -> Pipeline:
    preprocessor = make_preprocessor(X)
    return Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", model)
    ])


def evaluate_models_cv(
    X: pd.DataFrame,
    y: pd.Series,
    models: dict,
    n_splits: int = 5,
    seed: int = 42,
    return_oof: bool = True
) -> dict:
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    results = {}
    for name, model in models.items():
        pipe = make_pipeline(model, X)

        # cross_validate gives per-fold scores + timing
        cv_out = cross_validate(
            pipe, X, y,
            cv=cv,
            scoring="roc_auc",
            return_train_score=False,
            n_jobs=-1
        )
        fold_scores = cv_out["test_score"]
        mean_auc = float(np.mean(fold_scores))
        std_auc = float(np.std(fold_scores))

        oof_auc = None
        if return_oof:
            # For AUC we need scores (probabilities or decision function)
            # cross_val_predict supports method='predict_proba' or 'decision_function'.
            # We try predict_proba first; if not available, fall back to decision_function.
            try:
                oof_scores = cross_val_predict(
                    pipe, X, y, cv=cv, method="predict_proba", n_jobs=-1
                )[:, 1]
            except Exception:
                oof_scores = cross_val_predict(
                    pipe, X, y, cv=cv, method="decision_function", n_jobs=-1
                )
            oof_auc = float(roc_auc_score(y, oof_scores))

        results[name] = {
            "roc_auc_mean": mean_auc,
            "roc_auc_std": std_auc,
            "fold_scores": fold_scores,
            "oof_roc_auc": oof_auc,
            "fit_time_mean": float(np.mean(cv_out["fit_time"])),
            "score_time_mean": float(np.mean(cv_out["score_time"]))
        }

    return results


def print_cv_results(results: dict):
    rows = []
    for name, r in results.items():
        rows.append({
            "model": name,
            "roc_auc_mean": r["roc_auc_mean"],
            "roc_auc_std": r["roc_auc_std"],
            "oof_roc_auc": r["oof_roc_auc"],
            "fit_time_mean": r["fit_time_mean"]
        })
    summary = pd.DataFrame(rows).sort_values("roc_auc_mean", ascending=False)
    print(summary.to_string(index=False))

    print("\nPer-fold scores:")
    for name, r in results.items():
        print(f"\n{name}: {np.round(r['fold_scores'], 4)}")

In [38]:
from xgboost import XGBClassifier

X=train_df.drop(columns=target, inplace=False)
y=train_df[target]

In [50]:
#training of different ml models with their default parameters
models = {
    "RandomForest": RandomForestClassifier(random_state=random_state),
    "GradientBoosting": GradientBoostingClassifier(random_state=random_state),
    "GaussianNB": GaussianNB(),
    "SVC_RBF": SVC(kernel="rbf", C=1.0, gamma="scale", random_state=random_state),
    "XGBClassifier": XGBClassifier(random_state=random_state, eval_metric="auc")
}

results = evaluate_models_cv(X, y, models=models, n_splits=5, seed=random_state, return_oof=True)
print_cv_results(results)

           model  roc_auc_mean  roc_auc_std  oof_roc_auc  fit_time_mean
GradientBoosting      0.781357     0.005913     0.781312       5.696147
    RandomForest      0.764086     0.003214     0.764093       3.258994
   XGBClassifier      0.763548     0.007478     0.763371       0.866484
      GaussianNB      0.750834     0.006369     0.750775       0.074719
         SVC_RBF      0.720294     0.007484     0.720278      11.789201

Per-fold scores:

RandomForest: [0.762  0.7688 0.7606 0.762  0.7671]

GradientBoosting: [0.7849 0.783  0.7697 0.7838 0.7855]

GaussianNB: [0.7614 0.7547 0.7453 0.7447 0.748 ]

SVC_RBF: [0.7209 0.7215 0.7069 0.7301 0.7221]

XGBClassifier: [0.771  0.7652 0.7507 0.7704 0.7604]


In [51]:
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

os.makedirs("catboost_tmp", exist_ok=True)

models={
    "LGBMClassifier": LGBMClassifier(random_state=random_state, verbose=-1),
}
results = evaluate_models_cv(X, y, models=models, n_splits=5, seed=random_state, return_oof=True)
print_cv_results(results)

         model  roc_auc_mean  roc_auc_std  oof_roc_auc  fit_time_mean
LGBMClassifier       0.78148     0.005088     0.781379       2.188845

Per-fold scores:

LGBMClassifier: [0.7836 0.7823 0.7716 0.7862 0.7836]


The best result can be seen from the LGBMClassifier

In [53]:
def sanity_check_shuffle_y(
    X: pd.DataFrame,
    y: pd.Series,
    model,
    seed: int = 42
) -> float:
    """
    Shuffle target; AUC should drop to ~0.50. If not, suspect leakage/bug.
    """
    rng = np.random.default_rng(seed)
    y_shuffled = pd.Series(rng.permutation(y.values), index=y.index)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    pipe = make_pipeline(model, X)

    scores = cross_validate(pipe, X, y_shuffled, cv=cv, scoring="roc_auc", n_jobs=-1)["test_score"]
    return float(np.mean(scores))

# Optional leakage sanity check on your best model:
leak_auc = sanity_check_shuffle_y(X, y, model=models["LGBMClassifier"], seed=42)
print("Shuffle-y sanity AUC (should be ~0.50):", leak_auc)

Shuffle-y sanity AUC (should be ~0.50): 0.4983699411978968


In [55]:
# ---------- Optuna objective ----------
def make_objective(X: pd.DataFrame, y: pd.Series, n_splits=5, seed=42):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    def objective(trial: optuna.Trial) -> float:
        params = {
            "objective": "binary",
            "metric": "auc",
            "boosting_type": "gbdt",
            "random_state": 42,
            "verbosity": -1,

            # core
            "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.2, log=True),
            "n_estimators": trial.suggest_int("n_estimators", 400, 6000),
            "num_leaves": trial.suggest_int("num_leaves", 16, 512, log=True),
            "max_depth": trial.suggest_int("max_depth", -1, 16),
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 200),

            # sampling
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "subsample_freq": trial.suggest_int("subsample_freq", 0, 10),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),

            # regularization
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 1.0),
            "min_child_weight": trial.suggest_float("min_child_weight", 1e-3, 10.0, log=True),
        }

        model = LGBMClassifier(**params)

        pipe = make_pipeline(model, X)

        oof_scores = cross_val_predict(pipe, X, y, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
        auc = roc_auc_score(y, oof_scores)

        return auc

    return objective

In [56]:
# ---------- Run Optuna ----------
def tune_xgb_optuna(X, y, n_trials=50, seed=random_state):
    sampler = optuna.samplers.TPESampler(seed=seed)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(make_objective(X, y, n_splits=5, seed=seed), n_trials=n_trials)

    print("Best AUC:", study.best_value)
    print("Best params:", study.best_params)
    return study

X = train_df.drop(columns=[target])
y = train_df[target]

study = tune_xgb_optuna(X, y, n_trials=50, seed=42)

[I 2026-02-07 03:34:46,665] A new study created in memory with name: no-name-53dcca4e-8986-47ca-80d3-488f93b15d79
[I 2026-02-07 03:35:34,895] Trial 0 finished with value: 0.7646095754069371 and parameters: {'learning_rate': 0.00727491708802781, 'n_estimators': 5724, 'num_leaves': 201, 'max_depth': 9, 'min_child_samples': 35, 'subsample': 0.662397808134481, 'subsample_freq': 0, 'colsample_bytree': 0.9464704583099741, 'reg_alpha': 0.002570603566117598, 'reg_lambda': 0.023585940584142682, 'min_split_gain': 0.020584494295802447, 'min_child_weight': 7.579479953348009}. Best is trial 0 with value: 0.7646095754069371.
[I 2026-02-07 03:35:40,835] Trial 1 finished with value: 0.7757722399899973 and parameters: {'learning_rate': 0.0823143373099555, 'n_estimators': 1589, 'num_leaves': 29, 'max_depth': 2, 'min_child_samples': 64, 'subsample': 0.8099025726528951, 'subsample_freq': 4, 'colsample_bytree': 0.7164916560792167, 'reg_alpha': 0.0032112643094417484, 'reg_lambda': 1.8007140198129195e-07, 'm

Best AUC: 0.7866512216871505
Best params: {'learning_rate': 0.0015136219003485595, 'n_estimators': 3957, 'num_leaves': 144, 'max_depth': 10, 'min_child_samples': 151, 'subsample': 0.6329094669754594, 'subsample_freq': 3, 'colsample_bytree': 0.6581358204365774, 'reg_alpha': 8.742247036827536, 'reg_lambda': 0.010291916373512185, 'min_split_gain': 0.014361771444873508, 'min_child_weight': 0.018189058662490095}


In [57]:
best_params=study.best_params

In [59]:
def train_final_once(df: pd.DataFrame, target: str, best_params: dict, seed: int = 42):
    X = df.drop(columns=[target])
    y = df[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=seed
    )

    final_params = dict(best_params)
    final_params.update({
        "random_state": seed,
        "n_jobs": -1
    })

    model = LGBMClassifier(**final_params)
    pipe = make_pipeline(model, X_train)
    pipe.fit(X_train, y_train)

    test_scores = pipe.predict_proba(X_test)[:, 1]
    test_auc = roc_auc_score(y_test, test_scores)

    print("FINAL TEST AUC:", test_auc)
    return pipe, test_auc

final_pipe, test_auc = train_final_once(train_df, target, best_params=best_params, seed=random_state)

FINAL TEST AUC: 0.7837010637429606


c:\Users\User\all_project\projects_in_github\taiwan2005_credict_card_project\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [61]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)

best_model = XGBClassifier(
    **study.best_params,
    eval_metric="logloss",
    tree_method="hist",
    random_state=random_state,
    n_jobs=-1
)

pipe = make_pipeline(best_model, X)  # uses your make_pipeline + preprocessing

oof_proba = cross_val_predict(pipe, X, y, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]

print("OOF ROC AUC:", roc_auc_score(y, oof_proba))
print("OOF PR AUC (Average Precision):", average_precision_score(y, oof_proba))

OOF ROC AUC: 0.7867868755176016
OOF PR AUC (Average Precision): 0.5603348295131629


In [71]:
def pick_threshold_for_recall(y_true, proba_pos, target_recall=0.85):
    prec, rec, thr = precision_recall_curve(y_true, proba_pos)

    # thr length = len(prec)-1, so align:
    prec2, rec2 = prec[1:], rec[1:]

    mask = rec2 >= target_recall
    if not mask.any():
        # если заданный recall недостижим, берём максимум recall
        idx = np.argmax(rec2)
        t = thr[idx]
        return t, prec2[idx], rec2[idx]

    # среди порогов, где recall >= target, берём максимум precision
    idx = np.argmax(prec2[mask])
    t = thr[mask][idx]
    p = prec2[mask][idx]
    r = rec2[mask][idx]
    return t, p, r


# пример использования:
target_recall = 0.85
t, p, r = pick_threshold_for_recall(y, oof_proba, target_recall=target_recall)
print("Chosen threshold:", t)
print("Precision:", p)
print("Recall:", r)

Chosen threshold: 0.124387726
Precision: 0.32328080229226364
Recall: 0.8500659257864005


Chosing the threshold as recall and computing using the oof. I want the threshold to be around the 0.85

Note: since we are trying to balance and also maximize the recall the other metrics will drop such as precision for class 1 will drop

In [72]:
y_pred = (oof_proba >= t).astype(int)
tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()

print("TN FP FN TP:", tn, fp, fn, tp)
print("F1:", f1_score(y, y_pred))
print("Flag rate:", y_pred.mean())

TN FP FN TP: 9243 9448 796 4513
F1: 0.46839647119875455
Flag rate: 0.5817083333333334


In [73]:
threshold_recall=t

In [74]:
pred = (oof_proba >= threshold_recall).astype(int)

print("Confusion matrix:\n", confusion_matrix(y, pred))
print("Precision:", precision_score(y, pred))
print("Recall:", recall_score(y, pred))
print("F1:", f1_score(y, pred))
print("\nClassification report:\n", classification_report(y, pred))

Confusion matrix:
 [[9243 9448]
 [ 796 4513]]
Precision: 0.32325764630040826
Recall: 0.8500659257864005
F1: 0.46839647119875455

Classification report:
               precision    recall  f1-score   support

           0       0.92      0.49      0.64     18691
           1       0.32      0.85      0.47      5309

    accuracy                           0.57     24000
   macro avg       0.62      0.67      0.56     24000
weighted avg       0.79      0.57      0.60     24000



In [79]:
threshold={
    'threshold_recall': float(threshold_recall)
}

In [82]:
with open(THRESHOLD_PATH, "w", encoding='utf-8') as f:
    json.dump(threshold, f, indent=2)